<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/14_research_library.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q biopython sentence-transformers faiss-cpu
from google.colab import drive
drive.mount('/content/drive')

import json, os, time, numpy as np, pandas as pd
from tqdm import tqdm
BASE = '/content/drive/MyDrive/longevity_project'
os.makedirs(f'{BASE}/data/processed/library', exist_ok=True)
kd = json.load(open(f'{BASE}/data/processed/known_drugs.json'))
scores = pd.read_parquet(f'{BASE}/results/hallmark_scores_combined.parquet')
HALLMARKS = list(kd['by_hallmark'])

Mounted at /content/drive


In [3]:
known = {d for k in kd['by_hallmark'].values() for d in k['helpful'] + k['harmful']}
cands = scores[scores['is_named'] & ~scores.index.isin(known)]

short = {}
for hm in HALLMARKS:
    for d in cands[f'{hm}_pct'].nlargest(15).index:
        short.setdefault(d, []).append(hm)

shortlist = pd.DataFrame({'drug': list(short),
                          'hallmarks': [', '.join(v) for v in short.values()],
                          'n_hallmarks': [len(v) for v in short.values()]})
shortlist = shortlist.sort_values('n_hallmarks', ascending=False).reset_index(drop=True)
shortlist.to_csv(f'{BASE}/results/shortlist.csv', index=False)
print(len(shortlist), 'drugs on the shortlist')
shortlist.head(15)

73 drugs on the shortlist


,drug,hallmarks,n_hallmarks
0,sj-172550,"senescence, mitochondria, inflammation",3
1,tubacin,"senescence, mitochondria, inflammation",3
2,carbetocin,"senescence, mitochondria, inflammation",3
3,nitrazepam,"senescence, mitochondria, inflammation",3
4,as-605240,"senescence, inflammation",2
5,ku-c103655,"senescence, inflammation",2
6,ns-3694,"senescence, inflammation",2
7,lansoprazole,"senescence, inflammation",2
8,verteporfin,"senescence, inflammation",2
9,mycophenolate-mofetil,"nutrient_sensing, proteostasis",2


In [4]:
from Bio import Entrez
Entrez.email = 'your.email@example.com'   # use your own email, as in Session 4

TOPICS = ('(aging OR ageing OR senescence OR lifespan OR longevity OR mTOR OR autophagy OR mitochondria '
          'OR mitochondrial OR inflammation OR "protein folding" OR "unfolded protein" OR toxicity OR mechanism)')

def fetch(drug, n=40):
    q = f'"{drug.replace("-", " ")}"[Title/Abstract] AND {TOPICS}'
    ids = Entrez.read(Entrez.esearch(db='pubmed', term=q, retmax=n, sort='relevance'))['IdList']
    if not ids:
        return []
    recs = Entrez.read(Entrez.efetch(db='pubmed', id=','.join(ids), retmode='xml'))
    out = []
    for art in recs['PubmedArticle']:
        a = art['MedlineCitation']['Article']
        abstract = ' '.join(str(x) for x in a.get('Abstract', {}).get('AbstractText', []))
        if abstract:
            year = a.get('Journal', {}).get('JournalIssue', {}).get('PubDate', {}).get('Year', '')
            out.append({'pmid': str(art['MedlineCitation']['PMID']), 'title': str(a['ArticleTitle']),
                        'year': str(year), 'abstract': abstract})
    return out

for d in tqdm(shortlist['drug']):
    path = f'{BASE}/data/processed/library/{d}.json'
    if os.path.exists(path):
        continue
    try:
        json.dump(fetch(d), open(path, 'w'))
    except Exception as e:
        print(d, 'failed:', e)
    time.sleep(0.4)

100%|██████████| 73/73 [03:10<00:00,  2.60s/it]


In [5]:
rows = []
for d in shortlist['drug']:
    path = f'{BASE}/data/processed/library/{d}.json'
    if os.path.exists(path):
        rows += [{'drug': d, **a} for a in json.load(open(path))]

lib = pd.DataFrame(rows).drop_duplicates(['drug', 'pmid']).reset_index(drop=True)
counts = lib.groupby('drug').size().reindex(shortlist['drug'], fill_value=0)
lib.to_parquet(f'{BASE}/data/processed/library.parquet')

print(len(lib), 'abstracts in total')
print('Median per drug:', int(counts.median()))
print('No papers found:', ', '.join(counts[counts == 0].index) or 'none')

1728 abstracts in total
Median per drug: 35
No papers found: ku-c103655, ns-3694, aminopurvalanol-a, tgx-115, xmd-1499, parthenolide-(alternate-stereo), orthothymotinic-acid, nsc-632839, emf-bca1-16, sew-05685, rdr-01440sc, k784-3131, su-11652


In [6]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('NeuML/pubmedbert-base-embeddings',
                            device='cuda' if torch.cuda.is_available() else 'cpu')
texts = (lib['title'] + '. ' + lib['abstract']).tolist()
emb = model.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True).astype('float32')
np.save(f'{BASE}/data/processed/library_embeddings.npy', emb)
print(emb.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.33k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

(1728, 768)


In [7]:
import faiss
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
faiss.write_index(index, f'{BASE}/data/processed/library.faiss')
print(index.ntotal, 'abstracts indexed')

1728 abstracts indexed


In [8]:
QUESTIONS = {
    'mechanism': '{drug} mechanism of action and molecular targets',
    'aging': '{drug} effect on aging, lifespan, or {hallmark} in cells or animals',
    'safety': '{drug} toxicity, side effects, and safety',
}
HM_TEXT = {'senescence': 'cellular senescence', 'nutrient_sensing': 'mTOR and nutrient sensing',
           'mitochondria': 'mitochondrial function', 'autophagy': 'autophagy',
           'inflammation': 'inflammation', 'proteostasis': 'protein folding and proteostasis'}

def retrieve(drug, question, k=5):
    q = model.encode([question], normalize_embeddings=True).astype('float32')
    sims, idx = index.search(q, min(len(lib), 1000))
    hits = [(i, s) for i, s in zip(idx[0], sims[0]) if lib.loc[i, 'drug'] == drug][:k]
    return pd.DataFrame([{'pmid': lib.loc[i, 'pmid'], 'year': lib.loc[i, 'year'],
                          'match': round(float(s), 2), 'title': lib.loc[i, 'title'][:90]} for i, s in hits])

d = counts.idxmax()
hm = shortlist.set_index('drug').loc[d, 'hallmarks'].split(', ')[0]
print(f'Test drug: {d} (shortlisted for {hm})')
for kind, template in QUESTIONS.items():
    print(f'\n{kind.upper()}')
    print(retrieve(d, template.format(drug=d.replace('-', ' '), hallmark=HM_TEXT[hm])).to_string(index=False))

Test drug: tubacin (shortlisted for senescence)

MECHANISM
    pmid year  match                                                                                      title
27799547 2016   0.39 Discovery of selective small-molecule HDAC6 inhibitor for overcoming proteasome inhibitor 
28468311 2017   0.37 Tubacin, an HDAC6 Selective Inhibitor, Reduces the Replication of the Japanese Encephaliti
18472289 2008   0.35                 The aggresome pathway as a target for therapy in hematologic malignancies.
29133216 2018   0.34 Anti-cancer effects of naturally derived compounds targeting histone deacetylase 6-related
40284485 2025   0.31 Liposomal Tubacin: Strategies for the Formulation of a Highly Hydrophobic Anticancer Drug.

AGING
    pmid year  match                                                                                      title
31719145 2019   0.27 The histone deacetylase inhibitor tubacin mitigates endothelial dysfunction by up-regulati
29133216 2018   0.23 Anti-cancer effec

In [9]:
json.dump({'questions': QUESTIONS, 'hallmark_text': HM_TEXT, 'embedding_model': 'NeuML/pubmedbert-base-embeddings'},
          open(f'{BASE}/data/processed/library_config.json', 'w'), indent=2)

print(f"Built a research library for {len(shortlist)} shortlisted candidates (top 15 named, non-control "
      f"compounds per hallmark): {len(lib):,} PubMed abstracts (median {int(counts.median())} per drug; "
      f"{int((counts == 0).sum())} drugs with none), embedded with a PubMedBERT-based model and indexed with "
      f"FAISS for semantic search; retrieval tested on mechanism, aging, and safety questions.")

Built a research library for 73 shortlisted candidates (top 15 named, non-control compounds per hallmark): 1,728 PubMed abstracts (median 35 per drug; 13 drugs with none), embedded with a PubMedBERT-based model and indexed with FAISS for semantic search; retrieval tested on mechanism, aging, and safety questions.
